# LoRA 학습 실습 노트북

이 노트북은 `train_lora.py`의 핵심 흐름을 단계별로 따라가며, Colab에서 LoRA 학습 파이프라인을 이해하고 실행하는 데 초점을 둡니다.

## 0. 로컬 실행 준비

이 노트북은 VS Code/Jupyter 로컬 커널에서 바로 실행할 수 있도록 패키지 설치와 워크스페이스 경로 확인 셀을 먼저 배치합니다.

In [2]:
# 0) 로컬 실행 준비: 패키지 설치
!pip install -U -q transformers accelerate datasets peft bitsandbytes safetensors

173.18s - pydevd: Sending message related to process being replaced timed-out after 5 seconds



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [3]:
# 0) 워크스페이스 파일 경로 확인
from pathlib import Path

train_script = Path('/Users/acertainromance401/Desktop/AI_SYS/AI_SYS_TEAM/scripts/llm_poc/train_lora.py')
train_file = Path('/Users/acertainromance401/Desktop/AI_SYS/AI_SYS_TEAM/scripts/llm_poc/korean_instruct_200.jsonl')

print('train_lora.py exists:', train_script.exists())
print('korean_instruct_200.jsonl exists:', train_file.exists())
print('train_lora.py:', train_script)
print('korean_instruct_200.jsonl:', train_file)

train_lora.py exists: True
korean_instruct_200.jsonl exists: True
train_lora.py: /Users/acertainromance401/Desktop/AI_SYS/AI_SYS_TEAM/scripts/llm_poc/train_lora.py
korean_instruct_200.jsonl: /Users/acertainromance401/Desktop/AI_SYS/AI_SYS_TEAM/scripts/llm_poc/korean_instruct_200.jsonl


In [18]:
# 0) 로컬 환경 요약
import os
import sys

print('Python:', sys.version)
print('CWD:', os.getcwd())
print('Workspace contents sample:', os.listdir('/Users/acertainromance401/Desktop/AI_SYS/AI_SYS_TEAM')[:20])

Python: 3.10.13 (main, Sep 11 2023, 08:16:02) [Clang 14.0.6 ]
CWD: /Users/acertainromance401/Desktop/AI_SYS/AI_SYS_TEAM/scripts/llm_poc
Workspace contents sample: ['CODE_OF_CONDUCT.md', '스크린샷 2026-06-01 오후 4.01.45.png', '.venv-1', '.DS_Store', 'LICENSE', 'CHANGELOG.md', '.ruff_cache', 'code', 'docs', 'README.md', '.gitignore', 'CONTRIBUTING.md', '.venv', 'scripts', '.github', 'docker-compose.yml', 'Documents', '.git', 'assets', 'RETROSPECTIVE.md']


In [6]:
# 1) 필수 라이브러리 임포트 및 환경 점검
import os
import sys
import argparse

import datasets
import peft
import transformers

try:
    import accelerate
except Exception as exc:
    accelerate = None
    print(f'accelerate import failed: {exc}')

try:
    import bitsandbytes as bnb
except Exception as exc:
    bnb = None
    print(f'bitsandbytes import failed: {exc}')

print('Python:', sys.version)
print('datasets:', datasets.__version__)
print('transformers:', transformers.__version__)
print('peft:', peft.__version__)
print('accelerate:', getattr(accelerate, '__version__', 'not installed'))
print('bitsandbytes:', getattr(bnb, '__version__', 'not installed'))

Python: 3.10.13 (main, Sep 11 2023, 08:16:02) [Clang 14.0.6 ]
datasets: 5.0.0
transformers: 5.12.1
peft: 0.19.1
accelerate: 1.14.0
bitsandbytes: 0.49.2


## 2. 학습 인자 파싱 함수 구현

`argparse`를 사용해 모델 경로, 데이터 경로, 출력 디렉터리, 배치 크기, epoch, LoRA 하이퍼파라미터를 입력받는 함수를 정의합니다.

In [ ]:
# 2) 학습 인자 파싱 함수 구현

def parse_args(argv=None):
    parser = argparse.ArgumentParser()
    parser.add_argument('--model_name_or_path', required=True)
    parser.add_argument('--train_file', required=True)
    parser.add_argument('--output_dir', required=True)
    parser.add_argument('--target_modules', type=str, default=None)
    parser.add_argument('--per_device_train_batch_size', type=int, default=1)
    parser.add_argument('--gradient_accumulation_steps', type=int, default=8)
    parser.add_argument('--learning_rate', type=float, default=1e-4)
    parser.add_argument('--num_train_epochs', type=int, default=3)
    parser.add_argument('--max_length', type=int, default=512)
    parser.add_argument('--lora_r', type=int, default=8)
    parser.add_argument('--lora_alpha', type=int, default=32)
    parser.add_argument('--lora_dropout', type=float, default=0.05)
    return parser.parse_args(argv)

sample_args = argparse.Namespace(
    model_name_or_path='Qwen/Qwen2.5-0.5B-Instruct',
    train_file='/Users/acertainromance401/Desktop/AI_SYS/AI_SYS_TEAM/scripts/llm_poc/korean_instruct_expanded_700.jsonl',
    output_dir='/Users/acertainromance401/Desktop/AI_SYS/AI_SYS_TEAM/scripts/llm_poc/outputs/qwen25-05b-lora',
    target_modules=None,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    num_train_epochs=3,
    max_length=512,
    lora_r=8,
    lora_alpha=32,
    lora_dropout=0.05,
)
print(sample_args)

Namespace(model_name_or_path='Qwen/Qwen2.5-0.5B-Instruct', train_file='/Users/acertainromance401/Desktop/AI_SYS/AI_SYS_TEAM/scripts/llm_poc/korean_instruct_expanded_700.jsonl', output_dir='/Users/acertainromance401/Desktop/AI_SYS/AI_SYS_TEAM/scripts/llm_poc/outputs/qwen25-05b-lora', target_modules=None, per_device_train_batch_size=1, gradient_accumulation_steps=8, learning_rate=0.0002, num_train_epochs=1, max_length=512, lora_r=8, lora_alpha=32, lora_dropout=0.05)


## 3. JSONL 학습 데이터셋 로드

`datasets.load_dataset`으로 JSONL 파일을 읽고, `instruction`, `input`, `output` 필드가 있는 샘플 구조를 확인합니다.

In [20]:
# 3) JSONL 학습 데이터셋 로드
from datasets import load_dataset

train_file = '/Users/acertainromance401/Desktop/AI_SYS/AI_SYS_TEAM/scripts/llm_poc/korean_instruct_expanded_700.jsonl'
dataset = load_dataset('json', data_files={'train': train_file}, split='train')
print(dataset)
print(dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 700
})
{'instruction': '다음 판례를 한 문장으로 요약해줘.', 'input': '사건번호: 2021가합12345. 피고인이 슈퍼마켓에서 물건을 몰래 가져가서 소지하다가 적발되었다. 피고인은 도난 의도가 있었음을 인정했다.', 'output': '피고인이 슈퍼마켓에서 물건을 절취하여 유죄로 인정된 절도 사건.'}


## 4. 토크나이저 로드 및 패딩 토큰 설정

`AutoTokenizer`를 로드하고 `pad_token_id`가 없을 경우 `eos_token`을 패딩 토큰으로 설정합니다.

In [21]:
# 4) 토크나이저 로드 및 패딩 토큰 설정
from transformers import AutoTokenizer

model_name_or_path = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, use_fast=False)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print('pad_token:', tokenizer.pad_token)
print('eos_token:', tokenizer.eos_token)
print('pad_token_id:', tokenizer.pad_token_id)

pad_token: <|endoftext|>
eos_token: <|im_end|>
pad_token_id: 151643


## 5. 프롬프트 결합 및 토크나이징 전처리

`instruction`과 `input`을 프롬프트로 합치고 `output`을 이어 붙여 학습용 full sequence를 만든 뒤 토큰화하고 `labels`를 설정합니다.

In [31]:
# 5) 프롬프트 결합 및 토크나이징 전처리

def preprocess(example):
    instr = example.get('instruction', '') or ''
    inp = example.get('input', '') or ''
    out = example.get('output', '') or ''
    user_text = instr if not inp else instr + '\n\n' + inp

    prompt_messages = [
        {'role': 'system', 'content': '당신은 정확하고 간결하게 답하는 한국어 AI 어시스턴트입니다.'},
        {'role': 'user', 'content': user_text},
    ]
    full_messages = prompt_messages + [
        {'role': 'assistant', 'content': out},
    ]

    prompt_text = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    full_text = tokenizer.apply_chat_template(full_messages, tokenize=False, add_generation_prompt=False)

    prompt_encoding = tokenizer(prompt_text, truncation=True, max_length=512)
    full_encoding = tokenizer(full_text, truncation=True, max_length=512)

    labels = full_encoding['input_ids'].copy()
    prompt_len = min(len(prompt_encoding['input_ids']), len(labels))
    labels[:prompt_len] = [-100] * prompt_len
    full_encoding['labels'] = labels
    return full_encoding

preview = preprocess(dataset[0])
print('input_ids length:', len(preview['input_ids']))
print('labels length:', len(preview['labels']))
print('masked labels prefix:', preview['labels'][:10])

input_ids length: 133
labels length: 133
masked labels prefix: [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100]


## 6. 전처리된 데이터셋 생성

`dataset.map`으로 각 샘플에 전처리를 적용하고 원본 컬럼을 제거해 학습용 토큰화 데이터셋을 만듭니다.

In [11]:
# 6) 전처리된 데이터셋 생성
processed_dataset = dataset.map(preprocess, remove_columns=dataset.column_names, batched=False)
print(processed_dataset)
print(processed_dataset[0].keys())

Map:   0%|          | 0/137 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 137
})
dict_keys(['input_ids', 'attention_mask', 'labels'])


## 7. 4비트 모델 로드 및 K-bit 학습 준비

`AutoModelForCausalLM`을 4비트로 먼저 로드하고 실패 시 일반 정밀도로 대체하며, 필요 시 `prepare_model_for_kbit_training`을 적용합니다.

In [23]:
# 7) 4비트 모델 로드 및 K-bit 학습 준비
from transformers import AutoModelForCausalLM

use_4bit = False
base_model = None
try:
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name_or_path,
        load_in_4bit=True,
        device_map='auto',
    )
    use_4bit = True
    base_model = peft.prepare_model_for_kbit_training(base_model)
except Exception as exc:
    print('4-bit 로드 실패, 일반 정밀도로 로드합니다. (GPU 권장)')
    print(exc)
    base_model = AutoModelForCausalLM.from_pretrained(model_name_or_path, device_map='auto')

print(type(base_model))

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

4-bit 로드 실패, 일반 정밀도로 로드합니다. (GPU 권장)
Qwen2ForCausalLM.__init__() got an unexpected keyword argument 'load_in_4bit'


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

<class 'transformers.models.qwen2.modeling_qwen2.Qwen2ForCausalLM'>


## 8. LoRA 타깃 모듈 자동 탐지 로직 구현

`named_modules`를 순회해 `q_proj`, `k_proj`, `v_proj`, `o_proj`, `c_attn` 같은 후보 모듈을 탐지하고, 없으면 사용자가 `target_modules`를 지정하도록 합니다.

In [24]:
# 8) LoRA 타깃 모듈 자동 탐지 로직 구현

def collect_module_names(model):
    return [name for name, _ in model.named_modules()]


def auto_detect_target_modules(model):
    candidate_groups = [
        ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'down_proj', 'up_proj'],
        ['c_attn', 'c_proj'],
        ['query_key_value', 'dense'],
        ['qkv_proj', 'o_proj'],
    ]
    module_names = collect_module_names(model)
    lowered = [name.lower() for name in module_names]

    for group in candidate_groups:
        found = [candidate for candidate in group if any(candidate in name for name in lowered)]
        if found:
            return found

    primitives = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'c_attn', 'c_proj', 'query_key_value', 'dense']
    return [candidate for candidate in primitives if any(candidate in name for name in lowered)]


target_modules = auto_detect_target_modules(base_model)
if not target_modules:
    raise ValueError('target_modules를 자동 탐지하지 못했습니다. 수동으로 지정하세요.')
print('Using LoRA target modules:', target_modules)

Using LoRA target modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'down_proj', 'up_proj']


## 9. LoRA 설정 생성 및 PEFT 모델 래핑

`LoraConfig`를 생성하고 `get_peft_model`로 기본 모델을 LoRA 학습 가능 모델로 래핑합니다.

In [25]:
# 9) LoRA 설정 생성 및 PEFT 모델 래핑
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=target_modules,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
peft_model = get_peft_model(base_model, lora_config)
print(peft_model.__class__.__name__)

PeftModelForCausalLM


## 10. TrainingArguments와 Trainer 구성

배치 크기, gradient accumulation, learning rate, epochs, 저장 주기 등을 설정하고 `DataCollatorWithPadding`과 `Trainer`를 구성합니다.

In [36]:
# 10) TrainingArguments와 Trainer 구성
from transformers import TrainingArguments, Trainer
import torch
from dataclasses import dataclass

output_dir = '/Users/acertainromance401/Desktop/AI_SYS/AI_SYS_TEAM/scripts/llm_poc/outputs/qwen25-05b-lora'
use_fp16 = False
if torch.cuda.is_available():
    use_fp16 = not use_4bit
elif getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
    use_fp16 = False

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    num_train_epochs=3,
    fp16=use_fp16,
    save_total_limit=3,
    logging_steps=10,
    save_steps=50,
    remove_unused_columns=False,
)

@dataclass
class CausalLMDataCollator:
    tokenizer: object

    def __call__(self, features):
        labels = [feature.pop('labels') for feature in features]
        batch = self.tokenizer.pad(features, return_tensors='pt')
        max_len = batch['input_ids'].shape[1]
        padded_labels = []
        for label in labels:
            padded = label + [-100] * (max_len - len(label))
            padded_labels.append(padded)
        batch['labels'] = torch.tensor(padded_labels, dtype=torch.long)
        return batch


data_collator = CausalLMDataCollator(tokenizer)
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=processed_dataset,
    data_collator=data_collator,
)
print(trainer)

## 11. 학습 실행 및 LoRA 체크포인트 저장

`trainer.train()`으로 학습을 실행하고 `model.save_pretrained`로 LoRA 어댑터 체크포인트를 출력 디렉터리에 저장합니다.

다음 단계로는 LoRA 어댑터를 베이스 모델과 병합한 뒤 GGUF로 변환하면 iOS 앱에서 사용할 수 있습니다.

In [37]:
# 11) 학습 실행 및 LoRA 체크포인트 저장
trainer.train()
peft_model.save_pretrained(output_dir)
print('LoRA checkpoint saved to', output_dir)

import os
print('Output files:', os.listdir(output_dir))

Step,Training Loss
10,0.000000
20,0.000000
30,0.000000
40,0.000000
50,0.000000


LoRA checkpoint saved to /Users/acertainromance401/Desktop/AI_SYS/AI_SYS_TEAM/scripts/llm_poc/outputs/qwen25-05b-lora
Output files: ['adapter_model.safetensors', 'checkpoint-18', 'checkpoint-50', 'README.md', 'adapter_config.json', 'checkpoint-54']


## 12. LoRA 어댑터 병합

학습한 LoRA 어댑터를 베이스 모델에 병합해 단일 모델로 저장합니다.

In [ ]:
# 12) LoRA 어댑터 병합
from transformers import AutoModelForCausalLM
from peft import PeftModel

merged_output_dir = '/Users/acertainromance401/Desktop/AI_SYS/AI_SYS_TEAM/scripts/llm_poc/outputs/merged-model'
os.makedirs(merged_output_dir, exist_ok=True)

base_for_merge = AutoModelForCausalLM.from_pretrained(model_name_or_path, device_map='cpu')
adapter_model = PeftModel.from_pretrained(base_for_merge, output_dir, device_map='cpu')
merged_model = adapter_model.merge_and_unload()
merged_model.save_pretrained(merged_output_dir)
tokenizer.save_pretrained(merged_output_dir)
print('Merged model saved to', merged_output_dir)
print('Merged files:', os.listdir(merged_output_dir))

## 13. 병합 모델 추론 테스트

병합된 모델로 샘플 프롬프트를 생성해 학습 결과를 빠르게 확인합니다.

In [ ]:
# 13) 병합 모델 추론 테스트
import torch

prompt = '다음 판례를 한 문장으로 요약해줘.\n\n사건번호: 2021가합12345. 피고인이 슈퍼마켓에서 물건을 몰래 가져가서 소지하다가 적발되었다. 피고인은 도난 의도가 있었음을 인정했다.'
messages = [
    {'role': 'system', 'content': '당신은 정확하고 간결하게 답하는 한국어 AI 어시스턴트입니다.'},
    {'role': 'user', 'content': prompt},
]
chat_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(chat_prompt, return_tensors='pt')

source_length = inputs['input_ids'].shape[1]
device = next(peft_model.parameters()).device
inputs = {k: v.to(device) for k, v in inputs.items()}
peft_model = peft_model.eval()

with torch.no_grad():
    output_ids = peft_model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

new_tokens = output_ids[0][source_length:]
result = tokenizer.decode(new_tokens, skip_special_tokens=True)
print(result)